# **CS4120 Project: Analyzing Bias in Different Cuisine Type Restaurant Reviews (Yelp)**

### Melina Yang, Arpitha Coorg, Sheryl Cheng

## Yelp Restaurant & Cuisine Preprocessing

Filter businesses to restaurants, assign ethnic cuisine labels, merge reviews in chunks, balance samples, clean text, and export baseline data.

In [9]:
# imports
import gc
import os
import re
from pathlib import Path
import pandas as pd

In [10]:
# global variables
business_data = "yelp_academic_dataset_business.json"
review_data = "yelp_academic_dataset_review.json"

output_csv = "processed_reviews_baseline.csv"

review_chunk_size = 50_000
max_per_cuisine = 10_000
sample_random_state = 42

### Filter for restaurants only

Keep businesses whose `categories` string contains **Restaurants** (case-insensitive).

In [11]:
business_all = pd.read_json(business_data, lines=True)
n_business_total = len(business_all)

# case-insensitive: categories string must contain "Restaurants"
cat_mask = business_all["categories"].str.contains(
    "Restaurants", case=False, na=False, regex=False
)
business_rest = business_all.loc[cat_mask].copy()
n_restaurants = len(business_rest)

del business_all
gc.collect()

1048

### Assign cuisine types

Match comma-separated `categories` against keywords; first cuisine in the fixed priority order wins.

In [12]:
# order matters: first matching cuisine in this list is assigned.
cuisine_mapping = [
    ("Chinese", ["Chinese", "Cantonese", "Szechuan", "Dim Sum"]),
    ("Japanese", ["Japanese", "Sushi", "Ramen", "Izakaya"]),
    ("Korean", ["Korean"]),
    ("Thai", ["Thai"]),
    ("Vietnamese", ["Vietnamese", "Pho"]),
    ("Indian", ["Indian", "Pakistani", "Bangladeshi", "Himalayan"]),
    ("Mexican", ["Mexican", "Tex-Mex"]),
    ("Italian", ["Italian"]),
    (
        "American",
        ["American (Traditional)", "American (New)", "Burgers"],
    ),
    ("French", ["French"]),
]


def assign_cuisine_type(categories) -> str:
    """
    If any keyword appears in the categories string (case-insensitive),
    return the first matching cuisine in cuisine_mapping order.
    Otherwise return 'Other'.
    """
    if pd.isna(categories) or not str(categories).strip():
        return "Other"
    text = str(categories).lower()
    for cuisine, keywords in cuisine_mapping:
        for kw in keywords:
            if kw.lower() in text:
                return cuisine
    return "Other"


business_rest["cuisine_type"] = business_rest["categories"].map(assign_cuisine_type)

### Merge reviews with restaurant businesses (chunked)

Inner-merge each review chunk on `business_id`. Keep only rows where `cuisine_type != 'Other'`.

In [13]:
# slim lookup table for merges
biz_cols = ["business_id", "cuisine_type", "name", "city", "state"]
biz_lookup = business_rest[biz_cols].drop_duplicates(subset=["business_id"])
del business_rest
gc.collect()

review_keep = ["review_id", "user_id", "business_id", "stars", "text", "date"]
merge_cols = review_keep + ["cuisine_type", "name", "city", "state"]

merged_chunks: list[pd.DataFrame] = []

for i, chunk in enumerate(
    pd.read_json(review_data, lines=True, chunksize=review_chunk_size)
):
    chunk = chunk[review_keep]
    m = chunk.merge(biz_lookup, on="business_id", how="inner")
    m = m.loc[m["cuisine_type"] != "Other", merge_cols]
    if len(m) > 0:
        merged_chunks.append(m)

reviews_merged = pd.concat(merged_chunks, ignore_index=True)
del merged_chunks
gc.collect()

del biz_lookup
gc.collect()

0

### Balance: up to 10,000 reviews per cuisine

In [14]:
balanced_parts: list[pd.DataFrame] = []
for cuisine, g in reviews_merged.groupby("cuisine_type"):
    n = min(max_per_cuisine, len(g))
    balanced_parts.append(g.sample(n=n, random_state=sample_random_state))

balanced = pd.concat(balanced_parts, ignore_index=True)
del balanced_parts

del reviews_merged
gc.collect()

0

### Basic text cleaning and length filter

Lowercase, strip URLs, normalize whitespace; drop reviews with fewer than 10 words.

In [15]:
# match http(s) URLs and common www. spans
url_pattern = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
whitespace_pattern = re.compile(r"\s+")


def clean_review_text(text: str) -> str:
    if pd.isna(text):
        return ""
    t = str(text).lower()
    t = url_pattern.sub(" ", t)
    t = whitespace_pattern.sub(" ", t).strip()
    return t


balanced["text_clean"] = balanced["text"].map(clean_review_text)
balanced["word_count"] = balanced["text_clean"].str.split().str.len().fillna(0).astype(int)
balanced = balanced.loc[balanced["word_count"] >= 10].copy()
balanced = balanced.drop(columns=["word_count"])

### Save processed reviews CSV

In [16]:
save_cols = [
    "review_id",
    "user_id",
    "business_id",
    "stars",
    "text",
    "text_clean",
    "date",
    "cuisine_type",
    "name",
    "city",
    "state",
]
balanced[save_cols].to_csv(output_csv, index=False)

---

## Baseline Sentiment Analysis

Load `processed_reviews_baseline.csv`, score with **VADER** via `nltk.sentiment.vader` (`pip install nltk`). The VADER lexicon must already be downloaded into an NLTK data directory (e.g. `~/nltk_data` from `nltk.download('vader_lexicon')`); the setup cell prints where NLTK searches. Then summarize by cuisine and run correlation / ANOVA / pairwise tests.


In [ ]:
# VADER via NLTK — uses the vader_lexicon you downloaded with nltk.download
# (default location is usually ~/nltk_data on macOS; see printed paths below).
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

print("[NLTK] Corpora search paths (downloaded data lives under one of these):")
for _p in nltk.data.path:
    print(f"  {_p}")

def _vader_lexicon_available() -> bool:
    for rel in (
        "sentiment/vader_lexicon.zip",
        "corpora/vader_lexicon.zip",
        "sentiment/vader_lexicon/vader_lexicon.txt",
    ):
        try:
            nltk.data.find(rel)
            return True
        except LookupError:
            continue
    return False

if not _vader_lexicon_available():
    raise FileNotFoundError(
        "vader_lexicon not found in NLTK data paths. Download once in a terminal (with network), e.g.\n"
        '  python -c "import nltk; nltk.download(\'vader_lexicon\')"'
    )

sia = SentimentIntensityAnalyzer()
print("[VADER] SentimentIntensityAnalyzer ready (using downloaded NLTK data).")


### Load preprocessed data

In [2]:
df_sent = pd.read_csv("processed_reviews_baseline.csv")
display(df_sent.head(3))

,review_id,user_id,business_id,stars,text,text_clean,date,cuisine_type,name,city,state
0,of7o9WvF69sGcVVtv88v_Q,5o8TESbKhlQFRib6L0DHZQ,IDXLt6-LwcJTIFtKRJhW3g,5,"Exquisite staff; such a wonderful, perfect com...","exquisite staff; such a wonderful, perfect com...",2015-12-31 03:26:03,American,OTB Delight Café,Wesley Chapel,FL
1,uaSEIH9-3uW3YQwtMDnjMQ,ORZtHt-cu7GB83YtvT5iQw,AcGOk0jbFkvI1QzDpkqA8w,1,They never text me back I've been waiting for ...,they never text me back i've been waiting for ...,2019-02-23 05:59:13,American,The Rusty Lyon,Dunedin,FL
2,kE5U8qhhfKoYveEW6cRfDw,nJEvAVx0yYuF1pN3IinDiw,2hErWx-q_Ixx6zYgQMPw7g,5,Mangroves is one of the nicest restaurants in ...,mangroves is one of the nicest restaurants in ...,2013-05-29 14:43:33,American,Mangroves,Tampa,FL


### Apply VADER sentiment analysis

In [ ]:
texts = df_sent["text_clean"].fillna("").astype(str)
n = len(texts)
neg, neu, pos, compound = [], [], [], []

for i, t in enumerate(texts):
    if (i + 1) % 10_000 == 0 or (i + 1) == n:
        print(f"[VADER] Processing review {i + 1:,} / {n:,} ...")
    scores = sia.polarity_scores(t)
    neg.append(scores["neg"])
    neu.append(scores["neu"])
    pos.append(scores["pos"])
    compound.append(scores["compound"])

df_sent["vader_negative"] = neg
df_sent["vader_neutral"] = neu
df_sent["vader_positive"] = pos
df_sent["vader_compound"] = compound

print("\n[VADER] Sample: text_clean, stars, vader_compound (5 rows)")
sample_cols = ["text_clean", "stars", "vader_compound"]
for _, row in df_sent[sample_cols].head(5).iterrows():
    tc = row["text_clean"]
    if isinstance(tc, str) and len(tc) > 200:
        tc = tc[:200] + "..."
    print(f"  stars={row['stars']!r}  compound={row['vader_compound']:.4f}")
    print(f"  text: {tc!r}\n")

LookupError: 
**********************************************************************
  Resource 'vader_lexicon' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('vader_lexicon')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'sentiment/vader_lexicon.zip/vader_lexicon/vader_lexicon.txt'

  Searched in:
    - '/Users/sherylcheng/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/share/nltk_data'
    - '/Library/Frameworks/Python.framework/Versions/3.12/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
    - ''
**********************************************************************


In [ ]:
# === [SENTIMENT 4/8] Basic statistics by cuisine ===
try:
    from IPython.display import display
except ImportError:
    display = print

stat_cols = ["stars", "vader_compound"]

by_cuisine = df_sent.groupby("cuisine_type")[stat_cols].agg(
    ["mean", "std", "min", "max", "count"]
)
# Flatten MultiIndex columns -> stars_mean, stars_std, ...
by_cuisine.columns = [f"{a}_{b}" for a, b in by_cuisine.columns]
by_cuisine = by_cuisine.reset_index()

print("[STATS] By cuisine — sorted by mean stars (descending)")
display(by_cuisine.sort_values("stars_mean", ascending=False).reset_index(drop=True))

print("\n[STATS] By cuisine — sorted by mean vader_compound (descending)")
display(by_cuisine.sort_values("vader_compound_mean", ascending=False).reset_index(drop=True))

In [ ]:
# === [SENTIMENT 5/8] Sentiment–rating correlation ===
import numpy as np
from scipy import stats as scipy_stats

# Overall Pearson r (stars are discrete but widely used with Pearson here)
mask = df_sent["vader_compound"].notna() & df_sent["stars"].notna()
r_all, p_all = scipy_stats.pearsonr(
    df_sent.loc[mask, "vader_compound"],
    df_sent.loc[mask, "stars"],
)
print(f"[CORR] Overall Pearson r (vader_compound vs stars): r = {r_all:.4f}, p = {p_all:.2e}")

rows = []
for cuisine, g in df_sent.groupby("cuisine_type"):
    m = g["vader_compound"].notna() & g["stars"].notna()
    if m.sum() < 3:
        rows.append(
            {
                "cuisine_type": cuisine,
                "pearson_r": np.nan,
                "p_value": np.nan,
                "n": int(m.sum()),
                "note": "insufficient rows",
            }
        )
        continue
    r_c, p_c = scipy_stats.pearsonr(g.loc[m, "vader_compound"], g.loc[m, "stars"])
    rows.append(
        {
            "cuisine_type": cuisine,
            "pearson_r": r_c,
            "p_value": p_c,
            "n": int(m.sum()),
            "note": "",
        }
    )

corr_by_cuisine = pd.DataFrame(rows)
print("\n[CORR] Per cuisine (Pearson vader_compound vs stars)")
print(corr_by_cuisine.to_string(index=False))

In [ ]:
# === [SENTIMENT 6/8] ANOVA — star ratings across cuisines ===
groups = [
    g["stars"].values
    for _, g in df_sent.groupby("cuisine_type", sort=True)
    if len(g) >= 2
]
f_stat, p_anova = scipy_stats.f_oneway(*groups)

print("[ANOVA] One-way ANOVA on star ratings across cuisines")
print(f"  F-statistic: {f_stat:.4f}")
print(f"  p-value:     {p_anova:.2e}")
if p_anova < 0.05:
    print("  Interpretation: Significant differences exist between cuisines (p < 0.05).")
else:
    print("  Interpretation: No significant differences at α = 0.05.")

In [ ]:
# === [SENTIMENT 7/8] Pairwise Welch t-tests on stars ===
from itertools import combinations

cuisine_levels = sorted(df_sent["cuisine_type"].unique())
pair_rows = []
for c1, c2 in combinations(cuisine_levels, 2):
    a = df_sent.loc[df_sent["cuisine_type"] == c1, "stars"]
    b = df_sent.loc[df_sent["cuisine_type"] == c2, "stars"]
    t_stat, p_val = scipy_stats.ttest_ind(a, b, equal_var=False)
    mean_diff = float(a.mean() - b.mean())
    pair_rows.append(
        {
            "cuisine_a": c1,
            "cuisine_b": c2,
            "mean_diff_a_minus_b": mean_diff,
            "t_statistic": float(t_stat),
            "p_value": float(p_val),
            "abs_mean_diff": abs(mean_diff),
        }
    )

pair_df = pd.DataFrame(pair_rows)
top10 = pair_df.nlargest(10, "abs_mean_diff")

print("[PAIRWISE] Top 10 cuisine pairs by |mean star difference| (Welch t-test)")
cols_show = [
    "cuisine_a",
    "cuisine_b",
    "mean_diff_a_minus_b",
    "p_value",
    "t_statistic",
]
print(top10[cols_show].to_string(index=False))
print(
    "\n  (mean_diff_a_minus_b > 0 means cuisine_a has higher average stars than cuisine_b)"
)

In [ ]:
# === [SENTIMENT 8/8] Save CSV + summary report ===
out_csv = Path("processed_reviews_with_sentiment.csv")
out_txt = Path("baseline_sentiment_summary.txt")

df_sent.to_csv(out_csv, index=False)
print(f"[SAVE] Wrote {out_csv.resolve()}")

# Compact tables for the text report
stats_stars = by_cuisine.sort_values("stars_mean", ascending=False)[
    ["cuisine_type", "stars_mean", "stars_std", "vader_compound_mean"]
]
lines = [
    "Baseline sentiment analysis (VADER) — summary",
    "=" * 60,
    "",
    "OVERALL CORRELATION (Pearson: vader_compound vs stars)",
    f"  r = {r_all:.4f}",
    f"  p = {p_all:.2e}",
    "",
    "PER-CUISINE: mean stars, mean VADER compound, Pearson r (vs stars)",
    "-" * 60,
]
merge_corr = by_cuisine.merge(
    corr_by_cuisine[["cuisine_type", "pearson_r", "p_value"]],
    on="cuisine_type",
    how="left",
)
for _, row in merge_corr.sort_values("stars_mean", ascending=False).iterrows():
    lines.append(
        f"  {row['cuisine_type']}: "
        f"mean_stars={row['stars_mean']:.3f}, "
        f"mean_compound={row['vader_compound_mean']:.3f}, "
        f"r={row['pearson_r']:.4f} (p={row['p_value']:.2e})"
    )

lines.extend(
    [
        "",
        "ANOVA (star ratings across cuisines)",
        f"  F = {f_stat:.4f}",
        f"  p = {p_anova:.2e}",
        (
            "  Significant differences exist between cuisines (p < 0.05)."
            if p_anova < 0.05
            else "  No significant differences at alpha = 0.05."
        ),
        "",
        "TOP 10 PAIRWISE STAR-RATING DIFFERENCES (Welch t-test)",
        "-" * 60,
    ]
)
for _, row in top10.iterrows():
    lines.append(
        f"  {row['cuisine_a']} vs {row['cuisine_b']}: "
        f"mean_diff={row['mean_diff_a_minus_b']:+.4f}, p={row['p_value']:.2e}"
    )

lines.append("")
lines.append("Baseline sentiment analysis complete!")

out_txt.write_text("\n".join(lines), encoding="utf-8")
print(f"[SAVE] Wrote {out_txt.resolve()}")
print("\n>>> Baseline sentiment analysis complete! <<<")